In [13]:
# Celda 1 — Setup
import os
import sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/cuellojoaquinn/ThoraxNet.git /content/ThoraxNet
    !cp /content/drive/MyDrive/dataset.zip /content/
    !unzip -q /content/dataset.zip -d /content/dataset

    sys.path.append('/content/ThoraxNet/dev')
    DATASET_ROOT = "/content/dataset/data"
    OUTPUT_DIR   = "/content/drive/MyDrive/output"
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
    DATASET_ROOT = "e:/dataset/data"
    OUTPUT_DIR   = "e:/dataset/output"

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

Entorno: Local
DATASET_ROOT: e:/dataset/data
OUTPUT_DIR:   e:/dataset/output


In [14]:
from src.data              import get_loaders
from src.models.dannynet   import get_dannynet
from src.training.losses   import chexnet_loss, focal_loss
from src.training.train    import train_fine_tuning
from src.training.experiment import run_experiments, EXPERIMENTS
print("Imports OK")

Imports OK


In [ ]:
# 2. Verificar loaders
train_loader, val_loader, test_loader = get_loaders(
    DATASET_ROOT, batch_size=2, num_workers=4
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

In [ ]:
# 3. Verificar un batch
X, y = next(iter(train_loader))
print(f"X shape: {X.shape}")   # esperado: (128, 3, 224, 224)
print(f"y shape: {y.shape}")   # esperado: (128, 8)
print(f"y dtype: {y.dtype}")   # esperado: float32

In [ ]:
# 4. Verificar modelo
import torch
model = get_dannynet()
out   = model(X)
print(f"Output shape: {out.shape}")  # esperado: (128, 8)

In [ ]:
# 5. Verificar un solo step de entrenamiento
loss_fn   = chexnet_loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
X, y = X.to(device), y.float().to(device)
optimizer.zero_grad()
l = loss_fn(model(X), y)
l.backward()
optimizer.step()
print(f"Loss: {l.item():.4f} — un step OK")